# Tema 5: Fine-tuning with LoRA & QLoRA

Before diving in, let's import dependancies

In [1]:
!pip install -q -U "transformers>=4.41.0" "peft>=0.11.0" "trl>=0.9.4" \
                    "accelerate>=0.31.0" "bitsandbytes>=0.43.1" datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.2 MB/s eta 0:00:00


In [2]:
import torch
import random
import re
import pandas as pd

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


---

## 5A: Dataset

Here's the task:
> Use an open-source dataset for fine-tuning.
>
> Select a small evaluation set that is kept out of training.
>
> Use the same setup to compare the base and fine-tuned models.

We use **GSM8K**, an open-source grade-school mathematics dataset. Each example contains:
- `problem`: the question
- `gsm8k_answer`: the full solution and final answer
- `solution`: the final numeric answer

In [3]:
ds = load_dataset("juanlrdc/gmsk8")

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 2.33MB            

train.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B /  425kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [4]:
train_dataset = ds["train"]
test_dataset  = ds["test"]

In [ ]:
print(ds.keys())

dict_keys(['train', 'test'])


In [ ]:
print(f"Train dataset length: {len(train_dataset)}")
print(f"Test dataset length: {len(test_dataset)}")
print("\nFirst training example:")
print(train_dataset[0])
print("\nFirst test example:")
print(test_dataset[0])


Train dataset length: 7473
Test dataset length: 1319

First training example:
{'solution': ' 72', 'problem': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'id': 0, 'gsm8k_answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

First test example:
{'solution': ' 18', 'problem': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'id': 0, 'gsm8k_answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


---

## 5B: Fine-tuning with QLoRA

Here's the setup:
> Fine-tune **`Qwen/Qwen2.5-1.5B-Instruct`** using 4-bit QLoRA.
>
> Use the required LoRA configuration and train for 200 steps.
>
> Save the trained LoRA adapter separately from the base model.

**Configuration**
- Quantization: 4-bit, NF4, double quantization
- LoRA rank: 8
- LoRA alpha: 16
- Training steps: 200
- Batch size: 2
- Gradient accumulation: 16

**Reproducibility note:** a fixed training seed is set (via `set_seed`) before the `SFTTrainer` is created, so the run is reproducible. The `test` split is never touched during training (no monitoring, no checkpoint selection, no early stopping on it); it is only used afterwards, in section 5C, for evaluation.


In [5]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit QLoRA quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

print(model.config.architectures)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

['Qwen2ForCausalLM']


In [6]:
# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [7]:
# Format GSM8K examples with Qwen's native chat template.
def format_example(example):
    messages = [
        {"role": "user", "content": example["problem"].strip()},
        {"role": "assistant", "content": example["gsm8k_answer"].strip()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset_formatted = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names,
)

print(train_dataset_formatted[0]["text"])


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [ ]:
from transformers import set_seed

# Fixed seed for a reproducible training run.
TRAINING_SEED = 42
set_seed(TRAINING_SEED)

sft_config = SFTConfig(
    output_dir="./qlora-gsm8k",
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    dataset_text_field="text",
    max_length=512,
    packing=False,
    report_to="none",
    seed=TRAINING_SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_formatted,
)

train_result = trainer.train()
print(train_result)


Adding EOS to train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,0.920972
20,0.484706
30,0.464351
40,0.420233
50,0.423577
60,0.417535
70,0.400212
80,0.405490
90,0.396353
100,0.396279


TrainOutput(global_step=200, training_loss=0.429139461517334, metrics={'train_runtime': 9899.0973, 'train_samples_per_second': 0.647, 'train_steps_per_second': 0.02, 'total_flos': 1.268048944954368e+16, 'train_loss': 0.429139461517334, 'epoch': 0.8563018464008563})


---
### Export / Import QLoRa Adapter

In [ ]:
# Save only the LoRA adapter from the final training state.
ADAPTER_DIR = "./qlora-gsm8k-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to: {ADAPTER_DIR}")


Adapter saved to: ./qlora-gsm8k-adapter


In [ ]:
!zip -r ./QLoRa_adapter.zip ./qlora-gsm8k-adapter

from google.colab import files
files.download('QLoRa_adapter.zip')

  adding: qlora-gsm8k-adapter/ (stored 0%)
  adding: qlora-gsm8k-adapter/adapter_config.json (deflated 61%)
  adding: qlora-gsm8k-adapter/tokenizer.json (deflated 81%)
  adding: qlora-gsm8k-adapter/adapter_model.safetensors (deflated 21%)
  adding: qlora-gsm8k-adapter/chat_template.jinja (deflated 71%)
  adding: qlora-gsm8k-adapter/README.md (deflated 65%)
  adding: qlora-gsm8k-adapter/tokenizer_config.json (deflated 59%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import os
from peft import PeftModel

# Unzip the QLoRa_adapter
!unzip -o QLoRa_adapter.zip

# Load the base model again with the same quantization config
base_model_for_loading = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load the QLoRa adapter onto the base model
ADAPTER_DIR = "./qlora-gsm8k-adapter"
model_with_adapter = PeftModel.from_pretrained(
    base_model_for_loading,
    ADAPTER_DIR,
)

# Set the model to evaluation mode
model_with_adapter.eval()

print("QLoRa adapter loaded successfully.")

Archive:  QLoRa_adapter.zip
   creating: qlora-gsm8k-adapter/
  inflating: qlora-gsm8k-adapter/adapter_config.json  
  inflating: qlora-gsm8k-adapter/tokenizer.json  
  inflating: qlora-gsm8k-adapter/adapter_model.safetensors  
  inflating: qlora-gsm8k-adapter/chat_template.jinja  
  inflating: qlora-gsm8k-adapter/README.md  
  inflating: qlora-gsm8k-adapter/tokenizer_config.json  


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

QLoRa adapter loaded successfully.


---

## 5C: Evaluate Model Behavior

Here's the task:
> Compare the base model with the fine-tuned model on **20 questions total: 10 from the original `train` split and 10 from the original `test` split**.
>
> Compare their answers with the GSM8K reference answers.
>
> Report the results in a compact table, split accuracy by original data split.

Ground-truth values are read directly from the dataset's **`solution`** field (the final numeric answer), rather than parsed with a regex from `gsm8k_answer`. The model's free-text generations still need to be parsed to pull out a final number, so `extract_final_number` is kept for that purpose only; a small `normalize_number` helper makes sure both sides (dataset `solution` and parsed generation) are compared in the same format.


In [9]:
def generate_answer(gen_model, tokenizer, question, max_new_tokens=256):
    messages = [{"role": "user", "content": question.strip()}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def extract_final_number(text):
    """Extract the last number from a free-text model generation."""

    if text is None:
        return None

    if "####" in text:
        text = text.split("####")[-1]

    numbers = re.findall(
        r"-?\d[\d,]*\.?\d*",
        text.replace(",", ""))

    return numbers[-1] if numbers else None


def normalize_number(value):
    """Normalize a numeric value into a common string form."""

    if value is None:
        return None

    text = str(value).strip().replace(",", "")

    try:
        num = float(text)
    except ValueError:
        return text

    return str(int(num)) if num.is_integer() else str(num)


In [10]:
# Load the base model without the trained LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()

finetuned_model = model_with_adapter  # PEFT model trained above (adapter active)
finetuned_model.eval()

print("Base and fine-tuned models loaded.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base and fine-tuned models loaded.


In [ ]:
# 10 questions from the original train split + 10 questions from the original test split.
random.seed(123)

train_indices = random.sample(range(len(train_dataset)), 10)
test_indices = random.sample(range(len(test_dataset)), 10)

train_sample = train_dataset.select(train_indices)
test_sample = test_dataset.select(test_indices)

print(f"Train sample: {len(train_sample)} questions")
print(f"Test sample: {len(test_sample)} questions")


Train sample: 10 questions
Test sample: 10 questions


In [ ]:
results = []

for split_name, split_data in [("train", train_sample), ("test", test_sample)]:
    for example in split_data:
        question = example["problem"]
        gt_number = normalize_number(example["solution"])

        base_answer = generate_answer(base_model, tokenizer, question)
        ft_answer = generate_answer(finetuned_model, tokenizer, question)

        base_number = normalize_number(extract_final_number(base_answer))
        ft_number = normalize_number(extract_final_number(ft_answer))

        results.append({
            "split": split_name,
            "question": question,
            "ground_truth_number": gt_number,
            "base_answer": base_answer,
            "base_number": base_number,
            "base_correct": base_number == gt_number,
            "finetuned_answer": ft_answer,
            "finetuned_number": ft_number,
            "finetuned_correct": ft_number == gt_number,
        })

results_df = pd.DataFrame(results)
results_df


,split,question,ground_truth_number,base_answer,base_number,base_correct,finetuned_answer,finetuned_number,finetuned_correct
0,train,A frog lays her eggs over a series of 4 days. ...,810,To determine the total number of eggs laid by ...,810,True,The second day she laid 50*2=<<50*2=100>>100 e...,810,True
1,train,Suzanne wants to raise money for charity by ru...,310,To determine how much money Suzanne's parents ...,31,False,The total amount of money that Suzanne's paren...,150,False
2,train,John runs a website that gets 30000 visits a m...,10,To calculate how much John makes per day from ...,10,True,He makes .01*.3=<<.01*3=.03>>.03 dollars per v...,0.9,False
3,train,Ariel began fencing in 2006. If she was born i...,30,"To determine Ariel's current age, we need to f...",2008,False,She started at the beginning of 2006 so that’s...,16,False
4,train,"At the park, Dimitri saw families riding bicyc...",57,To determine the total number of wheels Dimitr...,57,True,There were 2 * 6 = <<2*6=12>>12 bicycle wheels...,57,True
5,train,Andrew installed hardwood flooring in his hous...,65,To determine how many wooden planks Andrew bou...,6,False,The guest bedroom took 8 - 2 = <<8-2=6>>6 plan...,50,False
6,train,An unusual bicycle has 20 spokes on the front ...,60,To determine the total number of spokes on the...,60,True,The number of spokes on the back wheel is 2*20...,60,True
7,train,William and Harry played 15 rounds of tic-tac-...,10,Let's denote the number of rounds Harry won as...,10,True,Let x be the number of rounds that Harry won.\...,10,True
8,train,Tommy is making steaks for his family. There a...,4,To determine how many steaks Tommy needs to bu...,4,True,Each person wants 1 pound which is equal to 16...,4,True
9,train,Summer and Jolly both went to the same middle ...,295,To determine the combined number of degrees th...,395,False,"If Summer has 150 degrees, then Jolly has 150-...",295,True


---


In [11]:
def generate_answers_batch(gen_model,tokenizer,questions,max_new_tokens=256):
    """
    Generate answers for multiple questions at once.
    """

    messages_list = [
        [{"role": "user", "content": question.strip()}]
        for question in questions
    ]

    prompts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in messages_list
    ]

    # Decoder-only models generally work best with left padding
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    # Some tokenizers don't have a pad token
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
    ).to(gen_model.device)

    with torch.inference_mode():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    # Restore tokenizer setting
    tokenizer.padding_side = old_padding_side

    # Because the inputs are padded to the same length,
    # generated tokens start after this common input length.
    input_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[:, input_length:]

    answers = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return [answer.strip() for answer in answers]


def evaluate_model_on_full_test_set(model,tokenizer,dataset,batch_size=64,max_new_tokens=256,desc="Evaluating"):

    was_training = model.training
    model.eval()
    rows = []

    # Process dataset in batches
    for start_idx in tqdm(
        range(0, len(dataset), batch_size),
        desc=desc,
        unit="batch",
    ):
        end_idx = min(start_idx + batch_size, len(dataset))

        batch = dataset.select(range(start_idx, end_idx))

        questions = batch["problem"]

        # Ground-truth numbers
        gt_numbers = [
            normalize_number(solution)
            for solution in batch["solution"]
        ]

        # Generate all answers in this batch at once
        answers = generate_answers_batch(
            model,
            tokenizer,
            questions,
            max_new_tokens=max_new_tokens,
        )

        # Process predictions
        for j, (question, gt_number, answer) in enumerate(
            zip(questions, gt_numbers, answers)
        ):
            predicted_number = normalize_number(
                extract_final_number(answer)
            )

            rows.append({
                "index": start_idx + j,
                "question": question,
                "ground_truth_number": gt_number,
                "answer": answer,
                "predicted_number": predicted_number,
                "correct": predicted_number == gt_number,
            })

    if was_training:
        model.train()

    results_df = pd.DataFrame(rows)
    accuracy = results_df["correct"].mean()

    return accuracy, results_df


base_test_accuracy, base_test_results = (
    evaluate_model_on_full_test_set(
        base_model,
        tokenizer,
        test_dataset,
        batch_size=64,
        max_new_tokens=256,
        desc="Evaluating base model",
    )
)

finetuned_test_accuracy, finetuned_test_results = (
    evaluate_model_on_full_test_set(
        finetuned_model,
        tokenizer,
        test_dataset,
        batch_size=64,
        max_new_tokens=256,
        desc="Evaluating fine-tuned model",
    )
)


print()
print(f"Base model accuracy:       {base_test_accuracy:.2%}")
print(f"Fine-tuned model accuracy: {finetuned_test_accuracy:.2%}")
print(f"Test questions evaluated:  {len(test_dataset)}")


Evaluating base model:   0%|          | 0/21 [00:00<?, ?batch/s]

Evaluating fine-tuned model:   0%|          | 0/21 [00:00<?, ?batch/s]


Base model accuracy:       40.56%
Fine-tuned model accuracy: 47.23%
Test questions evaluated:  1319
